# 03 - Which model catches fake job ads most usefully?

Compare the existing text-only logistic-regression model with A2 logistic regression, LightGBM and XGBoost.

Every model scores the **same ads in the same 25 rounds**: five groups of tests, repeated five times. Related ads always stay together. Each round chooses its settings and flagging cutoff using three smaller splits of that round's training ads. It then scores ads it has not seen. **The reserved final test file is never opened.**

Questions to answer:
1. Of the fake ads, how many does the model catch?
2. How many genuine ads does it wrongly flag?
3. If reviewers can read only the highest-scoring 5% of ads, how useful is that shortlist?
4. Does an improvement hold across repeats, and is the uncertainty small enough to support the claim?

A2 adds the original job fields, missing/contact/salary features and writing-style features. The three A2 models use identical features and a 2,000-term text limit. The existing text-only reference retains its 30,000-term limit. Compare the tree models with A2 logistic regression to isolate the change in model with matching inputs.

## 1. Run the comparison in this notebook

Notebook 02 is preliminary baseline work. This notebook reruns all four pipelines together for the main comparison, using shared outer folds and the same inner-fold procedure.

Select the project virtual-environment kernel, restart it to load the current source files, and run the cells in order. No terminal command is needed. The training cell shows progress below it; the full experiment may take many hours. Keep the computer awake and the kernel running.

The settings below use five repeats of five outer folds and three inner folds. All A2 models share a 2,000-term vocabulary; the text-only reference keeps 30,000 terms. Logistic regression tries C = 0.1 and 1.0, LightGBM tries 15 and 31 leaves, and XGBoost tries depths 3 and 5. This compares complete pipelines; the A2 models also provide a comparison with matching inputs.

New results go to `reports/model_comparison_local/`, preserving the earlier experiment. Completed models are reused only when the saved data, code and settings match. An interrupted partial model restarts. After completion, set `RUN_COMPARISON=False` to inspect results without training. If you later change code or settings, use a new output folder.


In [ ]:
import hashlib
import json
from pathlib import Path
import sys
from unittest.mock import patch

import pandas as pd
from IPython.display import Image, display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src' / 'model_comparison.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the project.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.model_comparison import MODEL_KEYS, MODEL_NAMES, main as run_comparison

REPORT_DIR = ROOT / 'reports' / 'model_comparison_local'
RUN_COMPARISON = True
N_REPEATS = 5
N_SPLITS = 5
INNER_SPLITS = 3
THREADS = 2

print(f'Results folder: {REPORT_DIR}')
print(f'{N_REPEATS} repeats x {N_SPLITS} outer folds; {INNER_SPLITS} inner folds.')


In [ ]:
# Progress appears below this cell. The full run may take many hours.
if RUN_COMPARISON:
    arguments = [
        'model_comparison', '--output', str(REPORT_DIR),
        '--repeats', str(N_REPEATS), '--splits', str(N_SPLITS),
        '--inner-splits', str(INNER_SPLITS), '--threads', str(THREADS),
    ]
    # Supply arguments directly to the Python entry point; no shell is used.
    with patch.object(sys, 'argv', arguments):
        run_comparison()
else:
    print('Training skipped; loading the completed comparison below.')


In [ ]:
missing = [key for key in MODEL_KEYS if not (REPORT_DIR / f'{key}_complete.json').exists()]
if missing:
    raise FileNotFoundError(f'Comparison is not complete yet. Missing models: {missing}. Run the training cell above first.')
plan = json.loads((REPORT_DIR / 'run_plan.json').read_text(encoding='utf-8'))
# Do not display an earlier experiment as though it used the current code/data.
for filename, expected in plan['source_sha256'].items():
    actual = hashlib.sha256((ROOT / 'src' / filename).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f'{filename} differs from this saved run. Restart the kernel and train into a new output folder.')
actual_data = hashlib.sha256((ROOT / 'data' / 'processed' / 'd1_train.csv').read_bytes()).hexdigest()
if actual_data != plan['training_sha256']:
    raise ValueError('Training data differs from this saved run. Train into a new output folder.')
summary = pd.read_csv(REPORT_DIR / 'comparison_summary.csv', index_col='model')
repeats = pd.read_csv(REPORT_DIR / 'results_by_repeat.csv')
paired = pd.read_csv(REPORT_DIR / 'paired_differences.csv')
print(f"{plan['training_rows']:,} ads: {plan['fraud_rows']} fake and {plan['genuine_rows']:,} genuine.")
print(f"{plan['repeats']} repeats x {plan['splits']} test rounds per model; {plan['inner_splits']} smaller training splits per round.")
print('The separate final test set has not been used.')

## 2. What happens to the same set of ads?

The ordinary flagging cutoff is chosen to balance catching fraud and avoiding false alarms, using training ads only. Counts below are **averages across the five repeats**. In each repeat, every training ad is scored once.

The 5% review results are a **different policy**: reviewers take the highest-scoring 5% within each held-aside batch, regardless of the ordinary cutoff. The percentage is rounded down to a whole number of ads in each batch.

The repeats reuse the same ads, so do not add the counts together as though they were five independent datasets.

In [ ]:
friendly = pd.read_csv(REPORT_DIR / 'plain_language_summary.csv', index_col=0)
display(friendly)
display(Image(filename=str(REPORT_DIR / 'comparison.png')))
print('Dots show the individual repeats. Error bars show their range, not a confidence interval.')

## 3. Is the pattern consistent across repeats?

Check whether a model catches more fraud only by creating more false alarms. The last two columns describe what happens when the review workload is held at about 5%.

In [ ]:
by_repeat = repeats[['model', 'repeat', 'fraud_caught', 'genuine_flagged', 'recall',
                     'false_positive_rate', 'review_count', 'review_fraud_caught',
                     'review_precision', 'review_recall', 'mean_fold_pr_auc']].copy()
by_repeat['model'] = by_repeat['model'].map(MODEL_NAMES)
by_repeat['repeat'] += 1
display(by_repeat.round(3))

## 4. Is a ranking improvement clear enough to trust?

PR-AUC measures how well fake ads are ranked ahead of genuine ads over many possible cutoffs. It is not the percentage of ads classified correctly.

The difference below is **candidate minus reference**. A positive PR-AUC difference favours the candidate. If its interval crosses zero, these tests do not clearly establish a ranking improvement. The intervals account approximately for overlap in training data; they are exploratory and are not adjusted for every comparison.

A better ranking does not automatically mean fewer false alarms at the selected cutoff. The other rows in `paired_differences.csv` compare fraud caught, false-alarm rates and the 5% review results.

In [ ]:
rank_comparison = paired[(paired['reference'] == 'logreg_tfidf') & (paired['metric'] == 'pr_auc')].copy()
rank_comparison['model'] = rank_comparison['model'].map(MODEL_NAMES)
display(rank_comparison[['model', 'mean_diff', 'low', 'high', 'share_folds_a_better']].round(3))
display(Image(filename=str(REPORT_DIR / 'paired_uncertainty.png')))
print('Same-feature comparison against A2 logistic regression:')
matched = paired[(paired['reference'] == 'logreg_a2') & paired['model'].isin(['lightgbm_a2','xgboost_a2'])]
display(matched[['model','metric','mean_diff','low','high','share_folds_a_better']].round(3))

## 5. Read the results in plain language

These statements describe the completed training-data tests. They do not choose a final model or use the reserved test set. Check whether the extra scams caught justify the extra genuine ads flagged for your intended review workload.

In [ ]:
for key in MODEL_KEYS:
    row = summary.loc[key]
    print(f"{MODEL_NAMES[key]}: catches {row['fraud_caught']:.1f} of {plan['fraud_rows']} fake ads "
          f"({row['recall']:.1%}), while wrongly flagging {row['genuine_flagged']:.1f} genuine ads "
          f"({row['false_positive_rate']:.1%} of genuine ads).")
    print(f"  With only 5% reviewed: {row['review_fraud_caught']:.1f} fake ads found; "
          f"{row['review_precision']:.1%} of reviewed ads are fake.")
    if key != 'logreg_tfidf':
        interval = paired[(paired['model']==key) & (paired['reference']=='logreg_tfidf') & (paired['metric']=='pr_auc')].iloc[0]
        if interval['low'] > 0:
            print('  These tests support better ranking than the text-only reference.')
        elif interval['high'] < 0:
            print('  These tests support worse ranking than the text-only reference.')
        else:
            print('  These tests do not clearly establish a ranking difference from the text-only reference.')
    print()

## Files for checking or reporting the work

- `reports/model_comparison_local/report.html`: standalone report with tables and figures.
- `comparison_summary.csv`: average outcomes across repeats.
- `results_by_repeat.csv`: counts and rates for each repeat.
- `paired_differences.csv`: approximate paired uncertainty for each metric.
- `repeat_differences_vs_baseline.csv`: changes relative to the reference in each repeat.
- `*_fold_scores.csv`, `*_inner_settings.csv`: round-by-round results and the settings tried.
- `run_plan.json`, `fold_plan.csv`: exact run settings, package versions, data/code fingerprints and test-round identities.
- `data/processed/model_comparison/`: exact row assignments and predictions, ignored by Git.

The next decision is which tradeoff is useful enough to take forward. Keep the final test file reserved until that decision and the final model settings have been fixed.